In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

# 1. Database Connection
load_dotenv("../.env")
DATABASE_URL = f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DATABASE_URL)

# 2. Load training data (Artıq loqlanmış y_train-i çəkirik)
X_train = pd.read_sql("SELECT * FROM X_train", engine)
y_train = pd.read_sql("SELECT * FROM y_train", engine).values.ravel()

# 3. Ordinal Mapping (Eyni qalır, amma 'None' dəyərlərini 0 etdiyiniz üçün əladır)
complex_mapping = {
    'exter_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'exter_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'bsmt_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'heating_qc': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'kitchen_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'fireplace_qu': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_qual': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'garage_cond': {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0},
    'pool_qc': {'Ex': 4, 'Gd': 3, 'TA': 2, 'Fa': 1, 'None': 0},
    'lot_shape': {'Reg': 4, 'IR1': 3, 'IR2': 2, 'IR3': 1},
    'land_slope': {'Gtl': 3, 'Mod': 2, 'Sev': 1},
    'bsmt_exposure': {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0},
    'garage_finish': {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0},
    'utilities': {'AllPub': 4, 'NoSewr': 3, 'NoSeWa': 2, 'ELO': 1},
    'functional': {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
}

def ordinal_encode(df, mapping):
    df = df.copy()
    for col, m in mapping.items():
        if col in df.columns:
            # Sütundakı dəyərləri map edirik, siyahıda olmayanları 0 edirik
            df[col] = df[col].map(m).fillna(0).astype(int)
    return df

X_train_encoded = ordinal_encode(X_train, complex_mapping)

# 4. Feature Selection
# 'neighborhood' kimi nominal sütunları bura daxil edirik
numeric_features = X_train_encoded.select_dtypes(include='number').columns.tolist()
nominal_features = X_train_encoded.select_dtypes(include='object').columns.tolist()

# 5. Pipeline Preprocessing
# DİQQƏT: StandardScaler-i rəqəmsal sütunlar üçün mütləq saxlayırıq
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), nominal_features)
])

# 6. XGBoost Model (Hyperparameters)
# learning_rate=0.05 və n_estimators=1000 kiçik datalarda overfit-in qarşısını almaq üçün yaxşıdır
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=1000, 
        learning_rate=0.05, 
        max_depth=6, 
        subsample=0.8, # Modelin hər dəfə datanın 80%-ni görməsi üçün (Overfit əleyhinə)
        colsample_bytree=0.8, 
        random_state=42,
        objective='reg:squarederror'
    ))
])

# 7. Cross-Validation
cv_scores = cross_val_score(final_pipeline, X_train_encoded, y_train, cv=5, scoring='r2')
print(f"Average CV R2: {cv_scores.mean():.4f}")

# 8. Training and Saving
final_pipeline.fit(X_train_encoded, y_train)

# Modeli saxlamaq üçün qovluğun olduğundan əmin olun
os.makedirs('../models', exist_ok=True)
joblib.dump(final_pipeline, '../models/ames_xgb_model.pkl')
print("Step 05: Model trained and serialized to /models/ames_xgb_model.pkl")